# OpenVINO Export: Depth Pro & SAM 2 Tiny

Export Depth Pro and SAM 2 vision encoders to OpenVINO IR format on Google Colab.
Run this notebook when your local machine lacks RAM for the tracing process.

**Note:** After running Cell 1, you may need to restart the runtime (Runtime → Restart runtime) before continuing, especially if Colab shows package conflicts.

In [ ]:
# Cell 1: Install dependencies
!pip install torch torchvision openvino openvino-dev optimum[intel] transformers huggingface_hub iopath hydra-core pillow

In [ ]:
# Cell 2: Clone SAM 2 and Depth Pro repos, install SAM 2
import os
import subprocess

# Clone SAM 2 (required for vision encoder export)
if not os.path.exists("sam2"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam2.git"], check=True)

# Clone Depth Pro original repo (optional reference; we use HuggingFace transformers for loading)
if not os.path.exists("ml-depth-pro"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/apple/ml-depth-pro.git"], check=True)

# Install SAM 2 (Python 3.10+ required on Colab)
os.environ["SAM2_BUILD_CUDA"] = "0"
subprocess.run(["pip", "install", "-q", "-e", "sam2"], check=True)

In [ ]:
# Cell 3: Export Depth Pro to OpenVINO FP16
import gc
import warnings
from pathlib import Path

import torch
import openvino as ov
from transformers import DepthProForDepthEstimation

torch.set_grad_enabled(False)

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)
depth_pro_dir = models_dir / "depth_pro_ov"
depth_pro_dir.mkdir(exist_ok=True)

print("=" * 60)
print("Exporting Depth Pro to OpenVINO FP16")
print("=" * 60)

model = DepthProForDepthEstimation.from_pretrained("apple/DepthPro-hf")
model.eval()

dummy_input = torch.zeros(1, 3, 1536, 1536)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    ov_model = ov.convert_model(model, example_input={"pixel_values": dummy_input})

ov.save_model(ov_model, depth_pro_dir / "openvino_model.xml", compress_to_fp16=True)
print(f"Saved to {depth_pro_dir}")

del model
del ov_model
gc.collect()

In [ ]:
# Cell 4: Export SAM 2 Tiny vision_encoder (image_encoder) to OpenVINO FP16
import gc
import warnings
from pathlib import Path

import torch
import openvino as ov
from sam2.sam2_image_predictor import SAM2ImagePredictor


class SamImageEncoderModel(torch.nn.Module):
    """Minimal wrapper for SAM 2 image encoder export (from OpenVINO notebooks)."""

    def __init__(self, predictor):
        super().__init__()
        self.image_encoder = predictor.model.image_encoder
        self.base_model = predictor.model
        self._bb_feat_sizes = predictor._bb_feat_sizes

    @torch.no_grad()
    def forward(self, image: torch.Tensor):
        backbone_out = self.base_model.forward_image(image)
        _, vision_feats, _, _ = self.base_model._prepare_backbone_features(backbone_out)
        if self.base_model.directly_add_no_mem_embed:
            vision_feats[-1] = vision_feats[-1] + self.base_model.no_mem_embed
        feats = [
            feat.permute(1, 2, 0).view(1, -1, *feat_size)
            for feat, feat_size in zip(vision_feats[::-1], self._bb_feat_sizes[::-1])
        ][::-1]
        # Return tuple of tensors (not list) for OpenVINO tracing
        return (feats[-1], feats[0], feats[1])


models_dir = Path("models")
sam2_dir = models_dir / "sam2_tiny_encoder_ov"
sam2_dir.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Exporting SAM 2 Tiny vision_encoder to OpenVINO FP16")
print("=" * 60)

predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2-hiera-tiny", device="cpu")
encoder_model = SamImageEncoderModel(predictor)
encoder_model.eval()

dummy_input = torch.zeros(1, 3, 1024, 1024)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    ov_encoder = ov.convert_model(
        encoder_model,
        example_input=dummy_input,
        input=([1, 3, 1024, 1024],),
    )

ov.save_model(ov_encoder, sam2_dir / "openvino_model.xml", compress_to_fp16=True)
print(f"Saved to {sam2_dir}")

del predictor
del encoder_model
del ov_encoder
gc.collect()

In [ ]:
# Cell 5: Zip models directory for easy download from Colab
import shutil
from pathlib import Path

models_dir = Path("models")
zip_path = Path("openvino_models.zip")

if models_dir.exists():
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", models_dir.parent, models_dir.name)
    print(f"Created {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
    print("Download from Colab: right-click openvino_models.zip -> Download")
else:
    print("models/ not found. Run cells 3 and 4 first.")